# Automatización Meme Reaction — Documentación

**Pipeline automatizado end-to-end para generar videos de meme reaction y subirlos a redes.**

Objetivo: Scraping de memes → Descarga → Clasificación IA → Match con clip de reacción → Verificación IA → Caption IA → Generación de video → (futuro: upload)

Todo este proyecto vive en: `automatizaciones/Meme_Reaction/`

## Pipeline General (Pasos)

```
1. SCRAPING (Selenium)
   Abre navegador → entra a IG → navega perfiles target → copia links de posts tipo foto

2. DESCARGA (instaloader, SIN LOGIN)
   Descarga las fotos usando los links obtenidos en paso 1

3. CLASIFICACIÓN IA (OpenAI Vision)
   Analiza cada meme y lo categoriza (tipo de humor/remate)

4. MATCH CON CLIP DE REACCIÓN
   Según la categoría del meme, elige un clip de reacción pre-catalogado

5. VERIFICACIÓN IA
   Verifica que la descripción del clip haga sentido con el meme

6. CAPTION IA
   Decide si vale la pena un caption y lo genera

7. GENERACIÓN DE VIDEO
   Meme arriba + Clip abajo + Caption (si aplica)
   Audio: SIEMPRE el del clip (no se puede cambiar en automatización)

8. GUARDAR JSON CONFIG
   Se guarda la combinación para replicar/editar manualmente después

9. (FUTURO) UPLOAD
   Subir a redes automáticamente
```

## `main.py` — Orquestador

**Qué hace:**
1. `pip install -r requirements.txt` (instala deps del sub-proyecto)
2. Carga `.env` del proyecto general (`../../.env`) con python-dotenv
3. Ejecuta los scripts numerados en orden: `1_` → `2_` → ... → `8_`
4. Si un script no existe (placeholder), lo salta con aviso
5. Al final muestra resumen de qué pasos pasaron y cuáles no

**Uso:**
```bash
python automatizaciones/Meme_Reaction/main.py
```

**Nota:** El `main.py` ejecuta cada paso como subprocess independiente. Cada `N_*.py` debe funcionar de forma standalone también (para debug individual).

---

## `requirements.txt` — Dependencias del Sub-Proyecto

Solo incluye lo que ESTE pipeline necesita (no todo el proyecto general):

| Grupo | Paquetes |
| --- | --- |
| Scraping | selenium, webdriver-manager |
| Instagram | instaloader |
| IA | openai, python-dotenv |
| Video | moviepy, Pillow, numpy |
| General | requests |

**Nota:** NO incluye yt-dlp, instagrapi, ni google-api (esos son del proyecto general, no de esta automatización).

## Paso 1: Scraping de Links con Selenium

**Archivo:** `1_scrape_meme_links.py` ✅ IMPLEMENTADO

**Navegador:** Brave (Chromium-based, usa ChromeDriver via webdriver-manager)

**Descubrimiento importante (mayo 2026):**
- Instagram ahora usa `/reel/` para CASI TODO en el grid (incluso posts que son fotos)
- Instagram virtualiza el DOM: los links desaparecen al scrollear fuera del viewport
- Es imposible distinguir fotos de videos desde el grid (no hay indicadores confiables)

**Solución:**
- Capturar TODOS los shortcodes (`/p/` y `/reel/`) durante el scroll
- Acumular en un Set conforme se scrollea (porque desaparecen del DOM)
- NO filtrar fotos vs videos aquí
- Dejar que el Paso 2 (instaloader) determine el tipo con `Post.from_shortcode().typename`

**Flujo:**
1. Abre Brave VISIBLE → navega a instagram.com
2. **PAUSA PARA LOGIN MANUAL** → espera Enter
3. Navega a cada perfil target
4. Scrollea N veces, capturando shortcodes EN CADA SCROLL (se acumulan)
5. Filtra contra historial (no repetir)
6. Guarda nuevos en `historial/links_scrapeados.json`

**Configuración:**
- `BRAVE_PATH` — ruta al ejecutable de Brave
- `PERFILES_TARGET` — lista de usernames a scrapear
- `SCROLL_COUNT` — cuántas veces scrollear (default: 8)
- `SCROLL_DELAY` — segundos entre scrolls (default: 3.0)

**Output:** `historial/links_scrapeados.json`
```json
{
  "scrapeados": ["shortcode1", "shortcode2", ...],
  "por_descargar": ["shortcode1", "shortcode2", ...]
}
```

**Nota:** Los shortcodes incluyen fotos Y videos mezclados. El filtrado se hace en el Paso 2.

## Paso 2: Descarga de Memes

**Archivo:** `2_download_memes.py` ✅ IMPLEMENTADO

**Comportamiento por tipo de post:**

| Tipo IG | Acción | Razón |
| --- | --- | --- |
| `GraphImage` | Descarga foto directa con `requests` | Es una foto simple |
| `GraphVideo` | Extrae primer frame con ffmpeg | Muchos son fotos con audio (no videos reales) |
| `GraphSidecar` | Skip | Es carousel, no lo queremos |
| `< 5000 likes` | Skip | Baja calidad / poco engagement |

**Importante:** NO usa `instaloader.download_post()` (tiene bug de paths en Windows). Solo usa `Post.from_shortcode()` para obtener tipo, URL y métricas, luego descarga con `requests` directo.

**Filtro de calidad (likes):**
- Default: **5000 likes mínimo** — si un post tiene menos, se salta sin descargar
- Se puede cambiar con `--min-likes N` o desactivar con `--min-likes 0`
- Los posts saltados por bajo engagement se registran en `skipped_low_likes` (con sus métricas)
- El request a IG sí se gasta (necesita el post para ver likes), pero no descarga el archivo

**Métricas guardadas:**
- Cada post descargado registra: `likes`, `comments`, `views` (solo videos)
- Se guardan en `posts_descargados.json` junto con shortcode y fecha
- Sirve para priorizar qué clasificar primero

**Protecciones anti-ban:**

| Parámetro | Valor | Propósito |
| --- | --- | --- |
| `MAX_POR_SESION` | 50 | Máximo requests por ejecución |
| `DELAY_ENTRE_POSTS` | 5s (+random 0-2s) | No spamear |
| `PAUSA_CADA_N` | 20 | Cada 20 posts, pausa larga |
| `PAUSA_DURACION` | 180s (3 min) | Enfriar la IP |
| `MIN_LIKES` | 5000 | No descargar basura |
| Detección rate limit | Auto-stop | Si IG responde 429 o pide login |

**Protección contra duplicados (NO gasta request a IG):**
- Antes de procesar, revisa `posts_descargados.json` + archivos existentes en `memes_descargados/`
- También salta los que ya están en `skipped_low_likes` o `skipped_carousels`
- Muestra: `[SKIP] N ya procesados (se sacan de la cola)`

**Distinción foto vs frame (importante para paso 3):**
- `GraphImage` → `descargados_foto` (con métricas)
- `GraphVideo` → `descargados_frame` (con métricas)

**Uso:**
```bash
python 2_download_memes.py               # Default: max 50, min 5000 likes
python 2_download_memes.py --max 10       # Solo 10 posts
python 2_download_memes.py --min-likes 0  # Sin filtro de likes
python 2_download_memes.py --min-likes 10000  # Solo virales
```

**Output:**
- Imágenes: `memes_descargados/{shortcode}.jpg`
- Log: `historial/posts_descargados.json`

**Estructura del log:**
```json
{
  "descargados_foto": [{"shortcode": "X", "fecha": "...", "likes": 12000, "comments": 45}],
  "descargados_frame": [{"shortcode": "Y", "fecha": "...", "likes": 8500, "comments": 120, "views": 250000}],
  "skipped_carousels": ["Z"],
  "skipped_low_likes": [{"shortcode": "W", "fecha": "...", "likes": 1200, "comments": 5, "views": 30000}],
  "errores": [{"shortcode": "A", "fecha": "..."}]
}
```

## Paso 3: Análisis Completo del Meme con IA

**Archivo:** `3_classify_meme.py` ✅ IMPLEMENTADO

**Filosofía:** La imagen ya se está pagando (85 tokens fijos con detail:low). Extraer TODO en UNA sola llamada para que los pasos siguientes NO necesiten re-analizar la imagen.

**Qué extrae por imagen (1 sola llamada a GPT-4o):**

| Campo | Tipo | Para qué sirve |
| --- | --- | --- |
| `valido` | bool | ¿Es realmente un meme? |
| `es_video_real` | bool | ¿El frame no funciona como meme estático? |
| `categorias` | array (1-3) | Match con clips de reacción (paso 4) |
| `confianza` | float | Qué tan seguro está |
| `descripcion` | string | Contexto completo para caption (paso 6) sin re-enviar imagen |
| `ideas_video` | array (2-3) | Ideas creativas de formato, caption y clip ideal |
| `background_color` | "negro"/"blanco"/"otro" | Elegir fondo del video final para que combine |
| `franjas_negras` | object | Instrucciones de crop programáticas (separado arriba/abajo) |
| `dia_especial` | null/string | Para publicar en el día correcto |

**Ideas de video (nuevo):**
Cada idea incluye:
- `formato`: meme_arriba_clip_abajo, dos_videos_paralelos, meme_con_caption_y_clip, otro
- `caption_sugerido`: texto para el video
- `clip_ideal`: descripción del clip que quedaría (puede referenciar memes populares: "Michael Jackson comiendo palomitas")
- `descripcion_idea`: por qué funciona

```json
"ideas_video": [
  {
    "formato": "dos_videos_paralelos",
    "caption_sugerido": "mi perro / yo",
    "clip_ideal": "izq: perro dormido, der: persona desvelada con pulgar arriba",
    "descripcion_idea": "Contraste visual del meme sobre viglar la casa"
  }
]
```

**Franjas negras (crop programático):**
```json
"franjas_negras": {
  "tiene": true,
  "arriba": 0.15,
  "abajo": 0.10,
  "crop_arriba": false,
  "crop_abajo": true
}
```
- Valores decimales (0.0-1.0) usables directo en código
- `crop_arriba` y `crop_abajo` evaluados INDEPENDIENTEMENTE
- Puede haber texto arriba (no cortar) pero abajo estar limpio (sí cortar)

**Días especiales:**
- `null` = atemporal
- `"viernes"`, `"lunes"` = día de la semana
- `"fin_de_ano"`, `"navidad"`, `"halloween"`, `"san_valentin"` = estacional
- Regla: "día 31" + "todo el año" = fin_de_ano (NO halloween)

**Categorías (1-3 por meme):**
humor_absurdo, humor_dark, cringe, sad_funny, wholesome, plot_twist, relatable, rage, sus, intellectual

**Costo estimado:** \~$0.008 por imagen (85 tokens imagen + \~600 tokens output). 20 imágenes ≈ $0.16

**Uso:**
```bash
python 3_classify_meme.py            # Clasifica hasta 20
python 3_classify_meme.py --max 5     # Solo 5 (para testing)
```

**QA:** `3.5_review_clasificacion.py` — abre imagen + muestra todo lo que dijo la IA, tu validas OK/MAL

## Revisión Manual: `revisar_memes.py`

**Archivo:** `revisar_memes.py` ✅ IMPLEMENTADO

**Propósito:** Filtrar memes ANTES de gastar tokens de OpenAI. Te muestra cada imagen descargada y tú decides si la conservas (pasa a clasificación IA) o la desechas (se borra del disco, no se clasifica).

**Flujo:**
1. Busca imágenes en `memes_descargados/` que NO estén ya en `descartados_manual.json` ni en `clasificaciones.json`
2. Para cada una:
   - Abre la imagen con el visor de Windows
   - Muestra tipo: "foto" o "frame (screenshot de video)"
   - Espera tu decisión
3. Guarda progreso después de cada decisión (puedes salir con `q` en cualquier momento)

**Controles:**

| Input | Acción |
| --- | --- |
| `Enter` o `s` | Mantener (pasa a paso 3) |
| `n` o `d` | Desechar (borra archivo, registra en historial) |
| `q` | Salir (guarda progreso) |

**Output:** `historial/descartados_manual.json`
```json
{
  "descartados": [{"shortcode": "ABC", "source_type": "frame (screenshot de video)", "fecha": "..."}],
  "mantenidos": [{"shortcode": "DEF", "source_type": "foto", "fecha": "..."}]
}
```

**Uso:**
```bash
python revisar_memes.py              # Revisa todos los pendientes
python revisar_memes.py --max 10     # Solo revisa 10
python revisar_memes.py --sin-abrir  # No abre visor (para terminales sin GUI)
```

**Cómo se integra al pipeline:**
```
Paso 2 (descarga) → revisar_memes.py (TÚ filtras) → Paso 3 (solo lo que mantuviste)
```

**Nota:** No importa si desechas DESPUÉS de clasificar — el paso 3 solo clasifica lo que EXISTE en `memes_descargados/`. Si borras el archivo, nunca se clasifica.

## Errores y Lecciones Aprendidas

Registro de problemas encontrados y sus soluciones, para no repetirlos.

### 1. Bug de paths de instaloader en Windows

**Problema:** `instaloader.download_post()` convierte rutas absolutas de Windows en nombres de carpeta con caracteres full-width (chinos): `C：﹨Users﹨David﹨...` en vez de `C:\Users\David\...`

**Síntoma:** Se creaba carpeta `C：﹨Users﹨David﹨Desktop﹨...` dentro del directorio actual (path anidado absurdo). El video se "descargaba" pero en una ubicación malformada, y luego no se encontraba para extraer el frame.

**Solución:** NO usar `instaloader.download_post()` para nada. Solo usar `Post.from_shortcode()` para obtener tipo y URL, y descargar con `requests.get(url)` directamente a paths que controlamos con `pathlib.Path`.

**Lección:** instaloader solo es confiable para queries/metadata, no para descargas en Windows.

---

### 2. ChromeDriver version mismatch (Brave 148 vs ChromeDriver 149)

**Problema:** `webdriver-manager` descargaba ChromeDriver 149 pero Brave era version 148. Selenium crasheaba con version mismatch.

**Solución:** Auto-detectar versión de Brave con PowerShell:
```python
(Get-Item 'C:\...\brave.exe').VersionInfo.FileVersion
```
Y pasar esa versión explícita a `ChromeDriverManager(driver_version=major_version)`.

**Lección:** Brave no se actualiza al mismo ritmo que Chrome. Siempre detectar versión real.

---

### 3. Python 3.11 - backslash en f-strings

**Problema:** `f"{'\n'.join(items)}"` causa SyntaxError en Python 3.11 (no permite backslash dentro de `{}` en f-strings).

**Solución:** Extraer a variable:
```python
separator = '\n'
result = f"{separator.join(items)}"
```

**Lección:** En Python 3.11, cualquier `\` dentro de `{}` de f-string es syntax error. Sacar a constante.

---

### 4. Instagram virtualiza el DOM (links desaparecen)

**Problema:** Selenium `find_elements()` después de scrollear devolvía solo 1 link. Los demás ya no estaban en el DOM.

**Solución:** 
- Capturar shortcodes EN CADA scroll (acumular en Set), no al final
- Usar `execute_script()` con `querySelectorAll()` en vez de `find_elements()`
- Capturar tanto `/p/` como `/reel/` (IG usa ambos para mismo contenido)

**Lección:** Instagram usa lazy rendering / virtual DOM. Los elementos desaparecen al salir del viewport. Siempre capturar durante el scroll, no después.

---

### 5. Instagram 403 no es error fatal

**Problema:** instaloader muestra `403 Forbidden when accessing graphql/query` pero internamente reintenta y funciona.

**Síntoma:** El log muestra el warning pero luego el post se procesa correctamente.

**Solución:** Ignorar el warning. Solo tratarlo como error real si `ConnectionException` llega al código (el retry interno falló).

**Lección:** El 403 es comportamiento normal de Instagram rate limiting suave. Instaloader ya maneja el retry internamente.

---

### 6. Todos los posts de cuentas de memes son GraphVideo

**Problema:** Se esperaba que muchos posts fueran `GraphImage`, pero las cuentas de memes publican CASI TODO como video (imagen + audio de fondo para el engagement/reach del algoritmo).

**Resultado real:** De 10 posts descargados, 9 fueron `GraphVideo` (frames) y 0 `GraphImage`.

**Impacto:** El flujo de "frame de video" es el caso más común, no la excepción. La mayoría de frames NO sirven como meme estático (son videos reales). Por eso `revisar_memes.py` es esencial antes de gastar en OpenAI.

**Lección:** No asumir que memes = fotos. En 2025-2026, Instagram incentiva video → las cuentas suben todo como reel. El paso de revisión manual es crítico para filtrar basura antes de la IA.

   
## Paso 4: Match Meme con Clip de Reacción (Interactivo + IA Conversacional)

**Archivo:** `4_match_clip.py` ✅ IMPLEMENTADO

**Modo:** INTERACTIVO — la IA sugiere, TÚ decides. Temporal mientras se llena la librería de clips.

**Flujo por meme:**
1. Abre la imagen del meme
2. La IA analiza el meme + catálogo de clips disponibles
3. Te muestra:
   - **Mejor match** del catálogo con % de accuracy (honesto)
   - **Caption sugerido** (o "sin caption" si el meme habla solo)
   - **Clip ideal** que deberías tener (con ejemplos virales concretos)
   - **Ideas alternativas** de formato
4. Tú decides: aceptar, cambiar caption, **conversar con la IA**, o skip

**Controles:**

| Input | Acción |
| --- | --- |
| `Enter` o `s` | Aceptar match (clip + caption) |
| `c` | Aceptar clip pero cambiar/quitar caption |
| `r` | **RESPONDER** - conversar con la IA (sugerir clip/caption) |
| `n` | Skip (voy a buscar un clip mejor por mi cuenta) |
| `q` | Salir |

### Modo Conversación (`r`)

Al presionar `r` entras en un chat libre con la IA donde:
- Le sugieres un clip + caption que crees que queda
- La IA evalúa tu idea con % de accuracy y feedback honesto
- Puedes iterar: "y si mejor sin caption?", "que tal el de Pedro Pascal?"
- Cuando la IA o tú deciden que está listo, te da un JSON final
- Confirmas y se guarda el match

**Ejemplo de conversación:**
```
Tu: creo que el de pedro pascal llorando queda bien con caption "cuando te gana la vida"
IA: El de Pedro Pascal llorando/riendo queda al 78% porque captura
    la dualidad del meme. El caption funciona pero podrías acortarlo
    a solo "la vida" para más impacto...
Tu: dale sin caption mejor
IA: Sin caption queda al 82% - el meme ya comunica solo y Pedro
    Pascal amplifica la emoción sin necesitar texto...
Tu: ok
-> MATCH ACEPTADO (via conversacion)
```

**Comandos dentro de conversación:**
- `ok` / `listo` / `dale` / `va` = Aceptar la última idea
- `salir` / `back` = Volver al menú principal (sin guardar)
- Cualquier otro texto = sigue la conversación

**Output:** `historial/matches.json`
```json
{
  "matched": [
    {
      "shortcode": "ABC123",
      "clip_id": "risa_01",
      "accuracy": 72,
      "caption": null,
      "clip_ideal_sugerido": "Pedro Pascal llorando/riendo",
      "conversacion": true,
      "fecha": "..."
    }
  ],
  "skipped_buscar_clip": [
    {
      "shortcode": "DEF456",
      "clip_ideal_sugerido": "Michael Jackson comiendo palomitas del video Thriller",
      "ideas": ["split screen", "..."],
      "fecha": "..."
    }
  ]
}
```

**Notas:**
- Campo `conversacion: true` indica que el match salió de la charla (no del análisis inicial)
- Incrementa `usado_count` del clip al aceptar match
- Los skipped quedan con la sugerencia de clip ideal (para saber qué buscar)
- Usa `gpt-4o-mini` (texto puro, prácticamente gratis)

**Uso:**
```bash
python 4_match_clip.py            # Procesa hasta 10
python 4_match_clip.py --max 3     # Solo 3 (para testing)
```

## Paso 5: Verificación IA (Meme + Clip)

**Funcionalidad:**
- Envía a la IA:
  - La imagen del meme
  - La descripción manual del clip elegido
  - La categoría asignada
- Pregunta: "¿Tiene sentido esta combinación? ¿El clip es buena reacción para este meme?"
- Si la IA dice que NO → se prueba otro clip de la misma categoría o se marca para revisión manual

**Output:** `aprobado` / `rechazado` + razón

**Estado:** POR IMPLEMENTAR

## Paso 6: Generación de Caption con IA

**Funcionalidad:**
- Envía a la IA:
  - La imagen del meme
  - La categoría
  - El clip elegido (descripción)
- Pregunta: "¿Este video necesita un caption superpuesto? Si sí, ¿cuál?"
- La IA puede responder:
  - `no_caption` — el meme habla por sí solo
  - `caption: "texto aquí"` — agregar este texto

**Reglas para el caption:**
- Corto (máximo 2 líneas)
- Usa `|` para salto de línea (patrón del generator)
- Puede ser meme text, reacción, o contexto

**Output:** caption string o None

**Estado:** POR IMPLEMENTAR

## Paso 7+8: Generación de Video + Config JSON

**Archivo:** `7_generate_video.py` ✅ IMPLEMENTADO

**Qué hace:** Toma cada match del paso 4 (que tenga clip asignado) y genera el video final. También guarda el JSON config automáticamente (paso 8 integrado).

**Flujo por match:**
1. Verifica que meme + clip existan
2. Pre-cropea franjas negras del meme (**PROGRAMÁTICO** - escanea pixels reales)
3. Auto-detecta barras negras del clip (letterboxing)
4. Calcula layout dinámico (meme 65-75% arriba, clip abajo)
5. Agrega caption si hay (tamaño auto-detectado por longitud)
6. Renderiza video 1080x1920 @ 30fps
7. Guarda config JSON en `configs_generados/`

**Inputs (de pasos anteriores):**

| Dato | Fuente |
| --- | --- |
| Meme imagen | `memes_descargados/{shortcode}.jpg` |
| Clip | `clips/{clip_id}.mp4` (copia local) |
| Caption | `matches.json` → campo `caption` |
| Background color | `clasificaciones.json` → `background_color` |

**Reglas:**
- Audio: SIEMPRE del clip (no se puede elegir externo en automatización)
- Auto-crop meme: PROGRAMÁTICO (escanea pixels, no depende de IA)
- Auto-crop clip: siempre habilitado (multiframe sampling)
- Caption size: auto-detectado (S >60 chars, M >30, L >15, XL <=15)
- Background: usa `background_color` de la clasificación (negro/blanco/otro)
- Duración: la del clip completo

**Controles:**

| Input | Acción |
| --- | --- |
| `Enter` o `s` | Generar video |
| `n` | Skip (no generar este) |
| `q` | Salir |
| `--auto` | Genera todo sin preguntar |

**Flag --redo (re-generar videos):**

Funciona igual que en el paso 3 — remueve shortcodes de `generados.json` (tanto de `generados` como de `errores`) para que vuelvan a ser "pendientes".

```bash
python 7_generate_video.py --redo ABC123          # Re-generar uno
python 7_generate_video.py --redo ABC123 DEF456   # Re-generar varios
python 7_generate_video.py --redo-all             # Re-generar TODOS
```

**Nota:** El `--redo` NO borra el video anterior de `output/`. Si regeneras, el nuevo video sobreescribe al anterior (mismo nombre de archivo).

**Uso normal:**
```bash
python 7_generate_video.py              # Interactivo (confirma cada uno)
python 7_generate_video.py --auto       # Genera todo sin preguntar
python 7_generate_video.py --max 3      # Solo 3 videos
```

**Output:**
- Videos: `output/meme_reaction/meme_{shortcode}_{clip_id}.mp4`
- Configs: `configs_generados/{shortcode}.json`
- Historial: `historial/generados.json`

**Config JSON generado:**
```json
{
  "shortcode": "ABC123",
  "meme_path": "automatizaciones/Meme_Reaction/memes_descargados/ABC123.jpg",
  "clip_path": "automatizaciones/Meme_Reaction/clips/risa_01.mp4",
  "clip_id": "risa_01",
  "caption": "cuando te gana la vida",
  "caption_size": "L",
  "categorias": ["relatable", "sad_funny"],
  "accuracy": 78,
  "audio_source": "clip",
  "auto_crop_clip": true,
  "auto_crop_meme": true,
  "background_color": "blanco",
  "output_name": "meme_ABC123_risa_01.mp4",
  "generated_at": "2026-05-20 18:30:00",
  "auto_generated": true,
  "conversacion": true
}
```

**Nota:** El paso 8 (guardar config) está integrado — se genera automáticamente junto con el video. No necesita script separado.

## ~~Paso 8: JSON Config~~ (Integrado en Paso 7)

El paso 8 ya no es un script separado. `7_generate_video.py` genera el JSON config automáticamente después de cada video.

Ver documentación del Paso 7 arriba.

---

## Pasos 5 y 6: Verificación + Caption (INNECESARIOS)

**Decisión:** Los pasos 5 (verificación IA) y 6 (caption IA) ya no son necesarios:
- La verificación se cubre con el `accuracy %` del paso 4 + la conversación interactiva
- El caption se genera/decide en el paso 4 (match) como parte de la sugerencia de la IA
- Gastar otra llamada a la API solo para verificar/caption sería redundante

**Pipeline final real:**
```
1. Scrape links → 2. Download → revisar_memes.py → 3. Clasificar IA
→ 4. Match con clip (interactivo/conversacional) → 7. Generar video + config
```

## Script Manual: Replicar/Editar desde JSON

**Archivo:** `manual_from_config.py` (por crear)

**Funcionalidad:**
- Carga un JSON config generado por la automatización
- Muestra qué tiene: meme, clip, caption, audio
- **Pregunta si quieres usar ese audio o elegir otro** (la diferencia con la automatización)
- Permite editar caption antes de generar
- Genera el video con los cambios

**Flujo:**
1. Lista JSONs disponibles en `configs_generados/`
2. Eliges uno
3. Te muestra preview de la combinación
4. "¿Usar audio del clip o elegir otro?" → si otro, usa browse_folder en audios/
5. "¿Editar caption?" → puedes cambiarlo o quitarlo
6. Genera video

**Estado:** POR IMPLEMENTAR

## Estructura de Carpetas

```
automatizaciones/Meme_Reaction/
├── DOCUMENTACION                # Este notebook
├── main.py                      # Orquestador: pip install + ejecuta pasos en orden
├── requirements.txt             # Dependencias SOLO de este sub-proyecto
├── 1_scrape_meme_links.py       # Paso 1: Selenium scraping de links
├── 2_download_memes.py          # Paso 2: Descarga con instaloader (sin login)
├── 3_classify_meme.py           # Paso 3: Clasificación IA del meme
├── 3.5_review_clasificacion.py  # QA: revisar clasificaciones de la IA
├── 4_match_clip.py              # Paso 4: Match meme → clip de reacción
├── 5_verify_match.py            # Paso 5: Verificación IA (meme+clip)
├── 6_generate_caption.py        # Paso 6: Generación de caption con IA
├── 7_generate_video.py          # Paso 7: Generar video final
├── 8_save_config.py             # Paso 8: Guardar JSON config
├── revisar_memes.py             # ★ Revisión manual (desechar/mantener antes de clasificar)
├── catalogar_clips.py           # ★ Catalogar clips de reacción (conversacional + IA)
├── manual_from_config.py        # Script manual para regenerar desde JSON
├── catalogo_clips.json          # Catálogo de clips con categorías y descripciones
├── config.json                  # Config general (por crear)
├── memes_descargados/           # Fotos/frames descargados (Paso 2)
│   └── {shortcode}.jpg
├── clips/                       # ★ Copias locales de clips catalogados
│   └── {clip_id}.mp4            # (el pipeline usa estos, no los originales)
├── configs_generados/           # JSONs de cada video generado (Paso 8)
│   └── .gitkeep
└── historial/                   # Control de estado y no-repetición
    ├── links_scrapeados.json    # Shortcodes capturados + cola por_descargar
    ├── posts_descargados.json   # Registro de descargas (foto vs frame vs error)
    ├── descartados_manual.json  # Memes descartados/mantenidos en revisión manual
    ├── clasificaciones.json     # Resultados de clasificación IA (Paso 3)
    └── review_clasificacion.json # QA de clasificaciones (OK/MAL + notas)
```

**Nota:** `.env` NO se duplica aquí. Se usa el del proyecto general (`../../.env`).

**Flujo de clips:**
```
tools_output/videos/ (todos los videos del proyecto)
        ↓ catalogar_clips.py (tú eliges + describes + IA mejora)
        ↓ COPIA a clips/ (local, con nombre = clip_id)
catalogo_clips.json apunta a clips/{clip_id}.mp4
        ↓ Paso 4/7 usa clips/ (nunca toca tools_output/ directamente)
```

## Config General (`config.json`)

Configuraciones del proyecto (propuesta):

```json
{
  "perfiles_target": [
    "elmello2023",
    "otro_perfil_memes"
  ],
  "max_posts_por_perfil": 10,
  "delay_entre_descargas": 5,
  "delay_entre_perfiles": 30,
  "selenium": {
    "headless": true,
    "scroll_count": 5,
    "scroll_delay": 2
  },
  "openai": {
    "model": "gpt-4o",
    "max_tokens": 500
  },
  "output_dir": "output/meme_reaction/",
  "caption_default_size": "M"
}
```

## Reglas del Proyecto

1. **NUNCA login en Instagram** — solo Selenium (scraping) + instaloader (descarga sin login)
2. **Audio en automatización = audio del clip SIEMPRE** — no se cambia
3. **Audio en manual (desde JSON) = se pregunta** — puede ser del clip o externo
4. **Cada cambio en código debe reflejarse en esta documentación**
5. **JSON config se guarda SIEMPRE** después de generar un video
6. **Clips se catalogan manualmente** — la IA no analiza videos, solo usa la descripción que yo escribo
7. **Historial de descargas** — nunca repetir un post ya descargado
8. **Verificación IA** — si no pasa, se busca otro clip o se marca para revisión manual

## Estado Actual

| Componente | Estado |
| --- | --- |
| Documentación | ✅ Creada y actualizada |
| Estructura de carpetas | ✅ Creada (.gitkeep en cada dir) |
| `main.py` (orquestador) | ✅ Creado (skeleton funcional) |
| `requirements.txt` | ✅ Creado |
| `1_scrape_meme_links.py` | ✅ Funcional (192 shortcodes en 1 corrida, Brave+login manual) |
| `2_download_memes.py` | ✅ Funcional (descarga + frames + likes + filtro 5K + sin 403 spam) |
| `revisar_memes.py` | ✅ Implementado (desechar/mantener con likes/views/tipo) |
| `3_classify_meme.py` | ✅ Implementado (GPT-4o Vision: categorías + descripción + ideas + franjas + día especial) |
| `3.5_review_clasificacion.py` | ✅ Implementado (QA de clasificaciones: OK/MAL + notas + accuracy) |
| `catalogar_clips.py` | ✅ Implementado (conversacional + IA mejora descripción + copia a clips/) |
| `4_match_clip.py` | ✅ Implementado (interactivo: IA sugiere match + caption + clip ideal viral) |
| `5_verify_match.py` | ⏳ Posiblemente innecesario (paso 4 ya verifica interactivamente) |
| `6_generate_caption.py` | ⏳ Posiblemente innecesario (paso 4 ya sugiere/confirma caption) |
| `7_generate_video.py` | ⏳ Por implementar |
| `8_save_config.py` | ⏳ Por implementar |
| `manual_from_config.py` | ⏳ Por implementar |
| `catalogo_clips.json` | ✅ Creado (clips catalogados con descripción IA) |
| `clips/` | ✅ Copias locales de clips catalogados |

**Próximos pasos:**
1. ✅ ~~Scraping~~ → ✅ ~~Descarga~~ → ✅ ~~Revisión manual~~ → ✅ ~~Clasificación IA~~ → ✅ ~~QA~~ → ✅ ~~Catalogar clips~~ → ✅ ~~Match~~
2. **Implementar paso 7** (generar video: meme + clip + caption)
3. **Implementar paso 8** (guardar JSON config)
4. Decidir si pasos 5/6 se eliminan (ya cubiertos por paso 3+4)
5. Implementar `manual_from_config.py` (regenerar con audio diferente)
6. Integrar todo via `main.py`
7. Seguir llenando librería de clips (ciclo: paso 4 skip → buscar → catalogar → re-match)

**Notas de testing:**
- Paso 3: prompt probado y validado (descripciones, ideas, franjas, días especiales OK)
- Paso 4: funciona con catálogo vacío (sugiere clip ideal aunque no tenga match)
- El 403 de instaloader ya no se muestra (stderr suprimido)
- Filtro de 5000 likes reduce basura significativamente
- `clips/` contiene copias locales — el pipeline nunca toca `tools_output/` directamente

## Setup Local (Primera Vez)

### 1. Prerrequisitos
- Python 3.10+ instalado
- **Brave Browser** instalado (ruta default: `C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe`)
- Git (si clonas el repo)

### 2. Clonar/Sincronizar el proyecto
Si ya tienes el repo en local, solo asegúrate de que la carpeta `automatizaciones/Meme_Reaction/` exista con todos los archivos.

### 3. Crear entorno virtual (recomendado)
```bash
cd drako-edits/automatizaciones/Meme_Reaction
python -m venv venv

# Windows:
venv\Scripts\activate

# Mac/Linux:
source venv/bin/activate
```

### 4. Instalar dependencias
```bash
pip install -r requirements.txt
```

### 5. Configurar `.env`
El `.env` vive en la raíz del proyecto general (`drako-edits/.env`). Debe tener:
```env
# OpenAI
OPENAI_API_KEY=sk-...

# Instagram (ya NO se usa login programático)
IG_USERNAME=
IG_PASSWORD=
```

### 6. Verificar Brave + Selenium
```bash
python -c "from selenium import webdriver; from webdriver_manager.chrome import ChromeDriverManager; print('OK')"
```
`webdriver-manager` descarga chromedriver automáticamente según tu versión de Brave.

### 7. Primer test
```bash
# Probar que el main corre (saltará todos los pasos porque son placeholder)
python main.py
```

### 8. Probar Selenium (Paso 1)
```bash
python 1_scrape_meme_links.py
```
Debería abrir Brave, navegar a IG, pausar para login, y luego scrapear.

---

### Orden de implementación para ir probando:
```
1. python 1_scrape_meme_links.py   ← probar Selenium con 1 perfil
2. python 2_download_memes.py      ← probar descarga de 1 foto
3. python 3_classify_meme.py       ← probar con 1 imagen local
4. python 4_match_clip.py          ← necesitas catalogo_clips.json primero
5. python 5_verify_match.py        ← probar con 1 combo
6. python 6_generate_caption.py    ← probar con 1 combo
7. python 7_generate_video.py      ← probar generación completa
8. python 8_save_config.py         ← verificar que grabe JSON
9. python main.py                  ← correr todo junto
```

Cada script debe funcionar **standalone** (lo corres individual para debug) Y también ser llamado por `main.py` en secuencia.